In [ ]:

import pandas as pd
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import GradientBoostingClassifier

import joblib
import os

In [ ]:
# abrindo os metadados
df = pd.read_csv("./../../../meta_dados/input_data/experiment_3/unsupervised_metric_sample_3.csv")
X = df.drop(columns=["target"]).to_numpy()
y = df["target"].to_numpy()

In [ ]:
# criando kfold
kf = KFold(n_splits=60, shuffle=True, random_state=42)

param_grid = {
    'loss': ['log_loss', 'exponential'],
    'learning_rate': [0.01, 0.1, 1],
    'n_estimators': [30, 50, 100, 200],
    'criterion': ['friedman_mse', 'squared_error'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_depth': [3, 5, 7],
    'min_impurity_decrease': [0.0, 0.1, 0.2],
    'max_features': ['sqrt', 'log2', None]
}

gb = GradientBoostingClassifier(random_state=42)


# Configurar o GridSearchCV
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    cv=kf, 
    scoring='accuracy', 
    n_jobs=-1,  
    verbose=1
)

# Executar o GridSearchCV
grid_search.fit(X, y)

Fitting 60 folds for each of 11664 candidates, totalling 699840 fits


In [ ]:
# Melhores parâmetros encontrados
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor score
print("Melhor acurácia:", grid_search.best_score_)

Scores: [0.38333333 0.35       0.30833333]
Média: 0.34722222222222227
Desvio padrão: 0.030681558381075724


In [ ]:
# Salvar modelo
joblib.dump(grid_search.best_estimator_, './saved_models/gb.joblib')

['./models_salvos/gb_clf.joblib']

Exception ignored in: <function ResourceTracker.__del__ at 0x7f3bb2a8ade0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7a21d3e8ede0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x79e236b8ede0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/reso

In [ ]:
# salvar resultados
def save_results(grid_search, filename="./resultados.csv"):
    """
    Verifica se o arquivo CSV existe, cria se não existir, e adiciona uma nova linha com model e best_score_.
    
    Args:
        grid_search (GridSearchCV): Objeto GridSearchCV treinado
        filename (str): Nome do arquivo CSV (padrão: './resultados.csv')
    """
    # Obtém o nome do modelo
    model_name = grid_search.estimator.__class__.__name__
    # Obtém o melhor score (neg_mean_squared_error, convertido para positivo se necessário)
    metric_score = grid_search.best_score_
    
    # Dados da nova linha
    new_data = pd.DataFrame({
        'model': [model_name],
        'metric_score': [metric_score]
    })
    
    # Verifica se o arquivo existe
    if not os.path.exists(filename):
        # Cria o arquivo com as colunas model e metric_score
        new_data.to_csv(filename, index=False)
    else:
        # Adiciona a nova linha ao arquivo existente
        existing_data = pd.read_csv(filename)
        updated_data = pd.concat([existing_data, new_data], ignore_index=True)
        updated_data.to_csv(filename, index=False)


In [ ]:
save_results(grid_search)